# 21. Python for Databases — the SQL APIs

*Module: 6 — Python access · Companion summary: `lectures/21-python-sql-apis-overview.md` · SQL engine: SQLite (in-memory)*

## What you will study

- name the steps of any SQL API flow (connect → cursor → execute → close)
- map Python's library options to their engines
- connect to a database from Python and run SQL

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 📗 API flow (language-neutral)

1. **Import** the library (or the generic module if the engine needs a connector).
2. **Connect** — open a connection object; provide credentials/path.
3. Get a **cursor**, or pass SQL directly to connection methods.
4. **Execute** SQL — the db engine receives and runs it.
5. **Close** the connection when done.

## 📗 Which library for which engine

| Engine | Library |
|---|---|
| SQLite (built into Python) | `sqlite3` |
| MySQL | `mysql-connector-python` / `pymysql` |
| PostgreSQL | `psycopg2` / `psycopg` |
| IBM DB2 | `ibm_db` |
| Oracle | `oracledb` |
| Any engine, one interface | `sqlalchemy` / DB-API `pyodbc` |

All follow the same flow below — learn the flow once, swap libraries often.

## 🧪 The full flow with sqlite3 — including the failure modes

In [ ]:
import sqlite3

DB = ':memory:'                  # in SQLite this could be a file path instead
conn = sqlite3.connect(DB)       # 2. connect
q_cur = conn.cursor()            # 3. cursor exists (also usable via conn.*)
q_cur.execute("CREATE TABLE demo (id INTEGER PRIMARY KEY, name TEXT)")        # 4. execute
q_cur.execute("INSERT INTO demo VALUES (1, 'hello')")
rows = q_cur.execute("SELECT * FROM demo").fetchall()                        # 5. read results
print('rows:', rows)
q_cur.close()
conn.close()                     # 5. always close — even after errors (try/finally!)

## 💡 Tips & tricks

- Wrap connection cleanup in `try/finally` (or a `with` block) — see nb22.
- Errors from the engine surface as **Python exceptions** (sqlite3.Error subclasses);
  reading them is a skill in itself (you'll meet several on purpose below).

## 🎯 Your turn

1. Re-run the flow but insert 2 rows and print both.
2. What happens if you call `q_cur.execute(...)` after close? Test it.

In [ ]:
# 💡 Solution
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.execute("CREATE TABLE demo (id INTEGER PRIMARY KEY, name TEXT)")
cur.execute("INSERT INTO demo VALUES (1, 'row one')")
cur.execute("INSERT INTO demo VALUES (2, 'row two')")
print('rows:', cur.execute("SELECT * FROM demo").fetchall())
cur.close()
try:
    cur.execute("SELECT 1")
except sqlite3.ProgrammingError as e:
    print('closed-cursor error:', e)
conn.close()

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

# re-establish the standard helper for the next cells